# Chapter 4.1: Coupling Geometry

This notebook isolates the Chapter 4 coupling-geometry experiments: independent versus OT endpoint coupling, Sinkhorn/minibatch sensitivity, rectified flow, and the coupling diagnostic table.


## 0. Setup

Imports, paths, device selection, random seeds, output directories, plotting style, cache helpers, and run-size controls.  The defaults below match the requested chapter settings; environment variables are only for smoke testing or resuming partial runs.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_ch04")

from pathlib import Path
import sys
import json
import math
import time
import random
import hashlib
import warnings
from dataclasses import dataclass
from typing import Callable, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    from torch import nn
except Exception as exc:
    raise ImportError("Chapter 4 experiments require PyTorch.") from exc

try:
    import anndata as ad
except Exception:
    ad = None

from scipy import sparse
from scipy.sparse.csgraph import shortest_path
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/home/xmabs/flow_matching_for_dynamic_biology/flow_matching_for_dynamic_biology")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import VelocityMLP as ProjectVelocityMLP, count_parameters
from src.losses import cfm_batch, cfm_loss_from_pairs
from src.train import train_cfm_steps
from src.sampling import euler_sample
from src.samplers import CouplingPairSampler as SrcCouplingPairSampler
from src.ot import (
    pairwise_squared_distances,
    independent_coupling,
    compute_ot_coupling_from_cost,
    sample_pair_indices_from_coupling,
    coupling_diagnostics,
)
from src.metrics import (
    mmd_rbf as project_mmd_rbf,
    sliced_wasserstein_distance,
    endpoint_metrics,
    trajectory_path_length,
    trajectory_path_energy,
    trajectory_straightness,
    off_manifold_knn_distance,
    fate_mass_error,
    coupling_l1_distance,
    normalized_cost_matrix,
    distribution_readout_metrics,
)
from src.representations import (
    fit_pca_state_space,
    pca_inverse_transform,
    program_index_dict,
    readout_program_scores_from_matrix,
    standardize_train_space,
    nearest_neighbor_overlap,
)

SEEDS = [42, 43, 44]
DEFAULT_SEED = 42
SOURCE_TIME = "1"
TARGET_TIME = "2"
TRAINING_STEPS = int(os.environ.get("CH04_TRAINING_STEPS", "1500"))
BATCH_SIZE = int(os.environ.get("CH04_BATCH_SIZE", "256"))
DEFAULT_NFE = int(os.environ.get("CH04_DEFAULT_NFE", "64"))
NFE_GRID = [2, 4, 8, 16, 32, 64]
SINKHORN_EPSILON = float(os.environ.get("CH04_SINKHORN_EPSILON", "0.05"))
EPSILON_GRID = [0.01, 0.02, 0.05, 0.1, 0.5]
BOOTSTRAP_REPEATS = int(os.environ.get("CH04_BOOTSTRAP_REPEATS", "50"))
EB_MAX_CELLS_PER_TIME = os.environ.get("CH04_EB_MAX_CELLS_PER_TIME", "")
EB_MAX_CELLS_PER_TIME = None if EB_MAX_CELLS_PER_TIME == "" else int(EB_MAX_CELLS_PER_TIME)
TOY_TRAINING_STEPS = int(os.environ.get("CH04_TOY_TRAINING_STEPS", str(TRAINING_STEPS)))
SMOKE_MODE = os.environ.get("CH04_SMOKE_MODE", "0") == "1"
if SMOKE_MODE:
    TRAINING_STEPS = min(TRAINING_STEPS, 20)
    TOY_TRAINING_STEPS = min(TOY_TRAINING_STEPS, 20)
    BATCH_SIZE = min(BATCH_SIZE, 64)
    DEFAULT_NFE = min(DEFAULT_NFE, 8)
    NFE_GRID = [2, 4, 8]
    BOOTSTRAP_REPEATS = min(BOOTSTRAP_REPEATS, 3)
    EB_MAX_CELLS_PER_TIME = 120 if EB_MAX_CELLS_PER_TIME is None else min(EB_MAX_CELLS_PER_TIME, 120)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = PROJECT_ROOT / "data"
EB_PATH = DATA_DIR / "trajectorynet_eb" / "eb_velocity_v5.npz"
TOY_DIR = DATA_DIR / "toy_branching_snapshots"
TOY_CSV_PATH = TOY_DIR / "observed_2d_snapshots.csv"
TOY_H5AD_PATH = TOY_DIR / "branching_toy_pseudocounts.h5ad"

FIG_DIR = PROJECT_ROOT /  "figures" / "ch04"
OUT_DIR = PROJECT_ROOT /  "outputs" / "ch04"
if SMOKE_MODE:
    FIG_DIR = FIG_DIR / "smoke"
    OUT_DIR = OUT_DIR / "smoke"
CACHE_DIR = OUT_DIR / "cache"
for path in [FIG_DIR, OUT_DIR, CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PALETTE = {
    "source": "#4C78A8",
    "target": "#F58518",
    "random": "#8E8E8E",
    "ot": "#008A7A",
    "reflow1": "#5369A6",
    "reflow2": "#B279A2",
    "rare": "#D95F02",
    "major": "#2C7FB8",
    "program": "#54A24B",
    "diagnostic": "#E45756",
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"Training steps: {TRAINING_STEPS}; batch size: {BATCH_SIZE}; default NFE: {DEFAULT_NFE}")
print(f"Smoke mode: {SMOKE_MODE}")


In [ ]:
def set_seed(seed: int = DEFAULT_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_ready(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(v) for v in obj]
    return obj


def save_json(path: str | Path, payload: dict) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_ready(payload), indent=2, sort_keys=True))
    return path


def load_json(path: str | Path):
    return json.loads(Path(path).read_text())


def save_npz(path: str | Path, **arrays) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **arrays)
    return path


def load_npz(path: str | Path):
    return np.load(Path(path), allow_pickle=True)


def save_csv(path: str | Path, frame: pd.DataFrame) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    return path


def save_pt(path: str | Path, payload) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)
    return path


def load_pt(path: str | Path, map_location=None):
    return torch.load(Path(path), map_location=map_location or DEVICE)


def save_figure(fig, filename: str, close: bool = True) -> Path:
    path = FIG_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    if close:
        plt.close(fig)
    return path


def artifact_exists(*paths) -> bool:
    return all(Path(p).exists() and Path(p).stat().st_size > 0 for p in paths)


def stable_hash(*items) -> str:
    h = hashlib.sha1()
    for item in items:
        h.update(str(item).encode())
    return h.hexdigest()[:10]


def sample_rows(n: int, max_n: int | None, seed: int = DEFAULT_SEED) -> np.ndarray:
    idx = np.arange(int(n))
    if max_n is None or n <= max_n:
        return idx
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(idx, size=int(max_n), replace=False))


def as_float32(x):
    return np.asarray(x, dtype=np.float32)


def to_tensor(x, device: torch.device = DEVICE):
    return torch.as_tensor(x, dtype=torch.float32, device=device)


def ensure_finite(name: str, x: np.ndarray) -> None:
    if not np.all(np.isfinite(x)):
        raise ValueError(f"{name} contains non-finite values")


## 1. Shared Utilities

The cells below define chapter-level wrappers and helpers.  They intentionally keep the key CFM/Sinkhorn/rollout/metric logic visible while reusing tested primitives from `src` where available.


In [ ]:
class VelocityMLP(ProjectVelocityMLP):
    """Chapter 4 VelocityMLP wrapper with requested defaults.

    The underlying implementation is imported from `src.models`, but the
    requested architecture is fixed here: hidden=128 and layers=4.
    """
    def __init__(self, input_dim: int, hidden: int = 128, layers: int = 4):
        super().__init__(x_dim=int(input_dim), hidden_dim=int(hidden), hidden_layers=int(layers))


def make_time_batch(batch_size: int, device: torch.device = DEVICE) -> torch.Tensor:
    return torch.rand(int(batch_size), 1, device=device)


def sample_independent_pairs(X0, X1, n_pairs: int, seed: int = DEFAULT_SEED):
    rng = np.random.default_rng(seed)
    i0 = rng.integers(0, len(X0), size=int(n_pairs))
    i1 = rng.integers(0, len(X1), size=int(n_pairs))
    return i0, i1


def compute_cost_matrix(x0, x1, normalize: bool = True):
    C = pairwise_squared_distances(np.asarray(x0, dtype=np.float32), np.asarray(x1, dtype=np.float32)).astype(np.float32)
    if not normalize:
        return C, 1.0
    positive = C[C > 0]
    scale = float(np.median(positive)) if positive.size else 1.0
    scale = max(scale, 1e-12)
    return (C / scale).astype(np.float32), scale


def sinkhorn_plan(C, epsilon: float = SINKHORN_EPSILON, return_info: bool = False):
    """Balanced Sinkhorn plan using POT when available, with project fallback."""
    return compute_ot_coupling_from_cost(np.asarray(C, dtype=np.float32), epsilon=float(epsilon), return_info=return_info)


def sample_from_plan(pi, n_pairs: int, seed: int = DEFAULT_SEED):
    return sample_pair_indices_from_coupling(np.asarray(pi, dtype=float), batch_size=int(n_pairs), seed=seed)


class PlanPairSampler:
    """Notebook-local thin wrapper around `src.samplers.CouplingPairSampler`."""
    def __init__(self, X0, X1, pi=None, seed: int = DEFAULT_SEED, labels0=None, labels1=None):
        self.X0 = as_float32(X0)
        self.X1 = as_float32(X1)
        if pi is None:
            pi = independent_coupling(len(self.X0), len(self.X1))
        self.pi = np.asarray(pi, dtype=float)
        self.src_sampler = SrcCouplingPairSampler(self.X0, self.X1, self.pi, seed=seed, labels0=labels0, labels1=labels1)

    def sample(self, batch_size: int = BATCH_SIZE):
        return self.src_sampler.sample(int(batch_size))


def train_cfm(
    model,
    pair_sampler,
    steps: int = TRAINING_STEPS,
    batch_size: int = BATCH_SIZE,
    lr: float = 1e-3,
    device: torch.device = DEVICE,
    seed: int = DEFAULT_SEED,
    log_every: int = 250,
):
    """Thin notebook wrapper around `src.train.train_cfm_steps`."""
    set_seed(seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=float(lr))
    history = train_cfm_steps(
        model,
        pair_sampler.sample,
        optimizer,
        n_steps=int(steps),
        batch_size=int(batch_size),
        device=device,
        log_every=int(log_every),
    )
    return history


@torch.no_grad()
def rollout_euler(model, x0, nfe: int = DEFAULT_NFE, device: torch.device = DEVICE):
    """Thin notebook wrapper around `src.sampling.euler_sample`."""
    model.eval()
    x0_t = to_tensor(x0, device)
    endpoint, _, _ = euler_sample(model, x0_t, n_steps=int(nfe), return_traj=False)
    return endpoint.detach().cpu().numpy().astype(np.float32)


@torch.no_grad()
def trajectory_rollout(model, x0, nfe: int = DEFAULT_NFE, return_path: bool = True, device: torch.device = DEVICE):
    """Thin notebook wrapper around `src.sampling.euler_sample`."""
    model.eval()
    x0_t = to_tensor(x0, device)
    endpoint, traj_t, _ = euler_sample(model, x0_t, n_steps=int(nfe), return_traj=return_path)
    endpoint_np = endpoint.detach().cpu().numpy().astype(np.float32)
    if return_path:
        traj_np = traj_t.detach().cpu().numpy().astype(np.float32)
        return endpoint_np, traj_np, np.linspace(0.0, 1.0, int(nfe) + 1)
    return endpoint_np


def mmd_rbf(X, Y, gamma: float | None = None):
    return project_mmd_rbf(np.asarray(X, dtype=float), np.asarray(Y, dtype=float), gamma=gamma)


def sliced_w2(X, Y, n_projections: int = 128, seed: int = DEFAULT_SEED):
    return sliced_wasserstein_distance(X, Y, n_projections=n_projections, seed=seed)


def path_length(traj):
    return trajectory_path_length(np.asarray(traj, dtype=float))


def path_energy(traj, times=None):
    return trajectory_path_energy(np.asarray(traj, dtype=float), times=times)


def tortuosity_straightness(traj):
    """Path-length/end-point tortuosity minus one; geometric, not biological validation."""
    return trajectory_straightness(np.asarray(traj, dtype=float))


def straightness(traj):
    """Backward-compatible alias used only inside this notebook."""
    return tortuosity_straightness(traj)


def straightness_action_S(traj, times=None):
    """Planning straightness action S = integral E[||(Z1-Z0)-dZ_t/dt||^2] dt.

    Euler trajectories provide finite-difference velocities. Lower S means
    finite-difference path velocities are closer to the endpoint chord.
    """
    traj = np.asarray(traj, dtype=float)
    if traj.ndim != 3 or traj.shape[0] < 2:
        raise ValueError("traj must have shape (T, N, D) with T >= 2")
    if times is None:
        times = np.linspace(0.0, 1.0, traj.shape[0])
    times = np.asarray(times, dtype=float)
    dt = np.diff(times)
    if len(dt) != traj.shape[0] - 1 or np.any(dt <= 0):
        raise ValueError("times must be strictly increasing and match traj")
    vel = np.diff(traj, axis=0) / dt[:, None, None]
    chord = traj[-1] - traj[0]
    sq = np.sum((chord[None, :, :] - vel) ** 2, axis=-1)
    return float(np.sum(sq.mean(axis=1) * dt))


def off_manifold_knn(points, reference, k: int = 15, batch_size: int = 1000):
    points_arr = np.asarray(points, dtype=np.float32)
    reference_arr = np.asarray(reference, dtype=np.float32)
    if points_arr.ndim == 3:
        points_arr = points_arr.reshape(-1, points_arr.shape[-1])
    if points_arr.ndim != 2 or reference_arr.ndim != 2:
        raise ValueError("points and reference must be 2D arrays, or points may be a (T, N, D) trajectory")
    if points_arr.shape[1] != reference_arr.shape[1]:
        raise ValueError("points and reference must have the same feature dimension")
    if points_arr.shape[0] <= batch_size:
        return off_manifold_knn_distance(points_arr, reference_arr, k=k)
    from sklearn.neighbors import NearestNeighbors
    kk = max(1, min(int(k), reference_arr.shape[0]))
    nn = NearestNeighbors(n_neighbors=kk, algorithm="ball_tree", leaf_size=40, n_jobs=1)
    nn.fit(reference_arr)
    total = 0.0
    count = 0
    for start in range(0, points_arr.shape[0], int(batch_size)):
        distances, _ = nn.kneighbors(points_arr[start:start + int(batch_size)])
        total += float(distances.sum())
        count += int(distances.size)
    return total / max(count, 1)

def coarse_step_error(model, x0, nfe_coarse: int = 4, nfe_fine: int = 64):
    z_coarse = rollout_euler(model, x0, nfe=nfe_coarse)
    z_fine = rollout_euler(model, x0, nfe=nfe_fine)
    return float(np.linalg.norm(z_coarse - z_fine, axis=1).mean())


def effective_support(pi):
    pi = np.asarray(pi, dtype=float)
    p = pi[pi > 0]
    return float(np.exp(-np.sum(p * np.log(p)))) if p.size else 0.0


def plan_entropy(pi):
    pi = np.asarray(pi, dtype=float)
    p = pi[pi > 0]
    return -float(np.sum(p * np.log(p))) if p.size else 0.0


def topk_nn_overlap(X_a, X_b, k: int = 15):
    return nearest_neighbor_overlap(X_a, X_b, k=k)


def coupling_topk_overlap(pi_a, pi_b, k: int = 15):
    pi_a = np.asarray(pi_a, dtype=float)
    pi_b = np.asarray(pi_b, dtype=float)
    if pi_a.shape != pi_b.shape:
        raise ValueError("coupling shapes differ")
    k = max(1, min(int(k), pi_a.shape[1]))
    rows = []
    for a, b in zip(pi_a, pi_b):
        ta = set(np.argpartition(-a, kth=k - 1)[:k].tolist())
        tb = set(np.argpartition(-b, kth=k - 1)[:k].tolist())
        rows.append(len(ta & tb) / float(k))
    return float(np.mean(rows))


def evaluate_endpoint(pred, target, seed: int = DEFAULT_SEED):
    return {
        "endpoint_mmd": mmd_rbf(pred, target),
        "sliced_w2": sliced_w2(pred, target, seed=seed),
    }


In [ ]:
def plot_phate_pairs(X0_phate, X1_phate, idx0, idx1, title: str, max_lines: int = 120, seed: int = DEFAULT_SEED, values=None):
    return plot_pair_panels(
        X0_phate,
        X1_phate,
        [{"title": title, "idx0": idx0, "idx1": idx1, "values": values, "seed": seed}],
        filename="_temporary_pair_panel.png",
        title=title,
    )


def plot_pair_panels(X0_phate, X1_phate, panels, filename: str, title: str, value_label: str = "PC-20 chord length"):
    from matplotlib.collections import LineCollection
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable

    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(5.0 * n, 4.2), squeeze=False)
    axes_flat = axes[0]
    all_values = []
    for panel in panels:
        if panel.get("values") is not None:
            all_values.append(np.asarray(panel["values"], dtype=float))
    norm = None
    if all_values:
        finite = np.concatenate([v[np.isfinite(v)] for v in all_values if np.any(np.isfinite(v))])
        norm = Normalize(vmin=float(finite.min()), vmax=float(finite.max())) if finite.size else None
    for ax, panel in zip(axes_flat, panels):
        idx0, idx1 = panel["idx0"], panel["idx1"]
        ax.scatter(X0_phate[:, 0], X0_phate[:, 1], s=8, color=PALETTE["source"], alpha=0.20, linewidths=0)
        ax.scatter(X1_phate[:, 0], X1_phate[:, 1], s=8, color=PALETTE["target"], alpha=0.20, linewidths=0)
        keep = sample_rows(len(idx0), min(panel.get("max_lines", 100), len(idx0)), seed=panel.get("seed", DEFAULT_SEED))
        segments = np.stack([X0_phate[idx0[keep]], X1_phate[idx1[keep]]], axis=1)
        values = panel.get("values")
        if values is not None and norm is not None:
            lc = LineCollection(segments, cmap="viridis", norm=norm, linewidths=0.8, alpha=0.55)
            lc.set_array(np.asarray(values, dtype=float)[keep])
            ax.add_collection(lc)
        else:
            lc = LineCollection(segments, colors=panel.get("color", "0.45"), linewidths=0.7, alpha=0.25)
            ax.add_collection(lc)
        ax.set_title(panel["title"])
        ax.set_xlabel("PHATE 1")
        ax.set_ylabel("PHATE 2")
    if norm is not None:
        fig.colorbar(ScalarMappable(norm=norm, cmap="viridis"), ax=list(axes_flat), fraction=0.035, pad=0.02, label=value_label)
    fig.suptitle(title, y=1.02)
    return save_figure(fig, filename)


def add_local_arrows(ax, projected_traj, observed_phate, color, max_arrows: int = 28, seed: int = DEFAULT_SEED):
    """Draw local average arrows only for trajectory points close to observed PHATE neighborhoods."""
    projected_traj = np.asarray(projected_traj, dtype=float)
    observed_phate = np.asarray(observed_phate, dtype=float)
    if projected_traj.shape[0] < 3 or projected_traj.shape[-1] != 2:
        return
    mid_step = projected_traj.shape[0] // 2
    anchors = projected_traj[mid_step]
    deltas = projected_traj[min(mid_step + 1, projected_traj.shape[0] - 1)] - projected_traj[max(mid_step - 1, 0)]
    nn_obs = NearestNeighbors(n_neighbors=min(15, len(observed_phate))).fit(observed_phate)
    obs_r = nn_obs.kneighbors(observed_phate, return_distance=True)[0][:, -1]
    dist = nn_obs.kneighbors(anchors, return_distance=True)[0][:, 0]
    threshold = float(np.quantile(obs_r, 0.75))
    valid = np.flatnonzero(dist <= threshold)
    if valid.size == 0:
        return
    keep = sample_rows(len(valid), min(max_arrows, len(valid)), seed=seed)
    valid = valid[keep]
    ax.quiver(
        anchors[valid, 0], anchors[valid, 1], deltas[valid, 0], deltas[valid, 1],
        angles="xy", scale_units="xy", scale=1.0, width=0.004, color=color, alpha=0.75,
    )


def plot_projected_trajectories(paths, X0_phate, X1_phate, pc_to_phate, filename: str, title: str, max_lines: int = 45, local_arrows: bool = True):
    n = len(paths)
    fig, axes = plt.subplots(1, n, figsize=(5.0 * n, 4.2), squeeze=False)
    observed_phate = np.vstack([X0_phate, X1_phate])
    for ax, (name, traj) in zip(axes[0], paths.items()):
        ax.scatter(X0_phate[:, 0], X0_phate[:, 1], s=8, color=PALETTE["source"], alpha=0.18, linewidths=0)
        ax.scatter(X1_phate[:, 0], X1_phate[:, 1], s=8, color=PALETTE["target"], alpha=0.16, linewidths=0)
        keep = sample_rows(traj.shape[1], min(max_lines, traj.shape[1]), seed=DEFAULT_SEED)
        selected = np.asarray(traj[:, keep, :], dtype=np.float32)
        flat = selected.reshape(-1, selected.shape[-1])
        ph = pc_to_phate(flat).reshape(selected.shape[0], selected.shape[1], 2)
        color = PALETTE.get(name, "0.35")
        for j in range(ph.shape[1]):
            ax.plot(ph[:, j, 0], ph[:, j, 1], color=color, alpha=0.55, linewidth=1.0)
        if local_arrows:
            add_local_arrows(ax, ph, observed_phate, color=color, seed=DEFAULT_SEED + 7)
        ax.set_title(name.replace("_", " "))
        ax.set_xlabel("PHATE 1 (display only)")
        ax.set_ylabel("PHATE 2 (display only)")
    fig.suptitle(title, y=1.02)
    return save_figure(fig, filename)


def plot_metric_lines(table, x: str, y: str, hue: str, filename: str, title: str):
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    for name, group in table.groupby(hue):
        group = group.sort_values(x)
        ax.plot(group[x], group[y], marker="o", linewidth=1.5, label=str(name).replace("_", " "))
    ax.set_xscale("log", base=2 if set(table[x]).issubset(set(NFE_GRID)) else 10)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title)
    ax.legend(frameon=False)
    return save_figure(fig, filename)


def plot_heatmap(matrix, title: str, filename: str, max_size: int = 120, cmap: str = "viridis"):
    M = np.asarray(matrix)
    rows = sample_rows(M.shape[0], min(max_size, M.shape[0]), seed=DEFAULT_SEED)
    cols = sample_rows(M.shape[1], min(max_size, M.shape[1]), seed=DEFAULT_SEED + 1)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(M[np.ix_(rows, cols)], aspect="auto", cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel("target subset")
    ax.set_ylabel("source subset")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return save_figure(fig, filename)


def plot_table_image(frame: pd.DataFrame, filename: str, title: str, max_rows: int = 12):
    shown = frame.head(max_rows).copy()
    fig, ax = plt.subplots(figsize=(min(14, 1.5 + 1.4 * shown.shape[1]), 0.8 + 0.35 * len(shown)))
    ax.axis("off")
    ax.set_title(title, loc="left")
    tbl = ax.table(cellText=shown.round(4).astype(str).values, colLabels=shown.columns, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1.0, 1.25)
    return save_figure(fig, filename)


## 2. Load EB Data

EB `pcs` is loaded from `eb_velocity_v5.npz`.  This file stores 100 PCs, so this notebook explicitly takes the first 20 PCs and standardizes them using all EB snapshots.  `phate` is kept as display-only coordinates.  The main EB bridge is time label `1 -> 2`.


In [ ]:
def load_eb_data(path: Path = EB_PATH, source_time: str = SOURCE_TIME, target_time: str = TARGET_TIME):
    if not path.exists():
        raise FileNotFoundError(path)
    z = np.load(path, allow_pickle=True)
    keys = list(z.files)
    pcs_full = np.asarray(z["pcs"], dtype=np.float32)
    phate = np.asarray(z["phate"], dtype=np.float32)[:, :2]
    labels = np.asarray(z["sample_labels"]).astype(str)
    pcs20_raw = pcs_full[:, :20].astype(np.float32)
    mean = pcs20_raw.mean(axis=0)
    std = pcs20_raw.std(axis=0)
    std = np.where(std < 1e-6, 1.0, std)
    pcs20 = ((pcs20_raw - mean) / std).astype(np.float32)
    available = sorted(np.unique(labels).tolist(), key=lambda s: int(s) if str(s).isdigit() else str(s))
    if str(source_time) not in available or str(target_time) not in available:
        raise ValueError(f"Requested bridge {source_time}->{target_time}; available labels: {available}")
    idx_source_all = np.flatnonzero(labels == str(source_time))
    idx_target_all = np.flatnonzero(labels == str(target_time))
    if EB_MAX_CELLS_PER_TIME is not None:
        idx_source = idx_source_all[sample_rows(len(idx_source_all), EB_MAX_CELLS_PER_TIME, seed=DEFAULT_SEED)]
        idx_target = idx_target_all[sample_rows(len(idx_target_all), EB_MAX_CELLS_PER_TIME, seed=DEFAULT_SEED + 1)]
    else:
        idx_source, idx_target = idx_source_all, idx_target_all
    counts = pd.Series(labels, name="time").value_counts().sort_index().rename_axis("time").reset_index(name="n_cells")
    summary = {
        "source_path": str(path),
        "available_keys": keys,
        "pcs_shape_actual": list(pcs_full.shape),
        "pc20_shape_used": list(pcs20.shape),
        "phate_shape": list(phate.shape),
        "sample_label_values": counts.to_dict(orient="records"),
        "source_time": str(source_time),
        "target_time": str(target_time),
        "n_source_full": int(len(idx_source_all)),
        "n_target_full": int(len(idx_target_all)),
        "n_source_used": int(len(idx_source)),
        "n_target_used": int(len(idx_target)),
        "training_space": "standardized PC-20 from pcs[:, :20]",
        "ot_cost_space": "standardized PC-20 unless Exp 7 contrastive diagnostic",
        "display_space": "PHATE 2D only for visualization",
        "distributional_metrics_space": "standardized PC-20",
        "standardization": "mean/std fit on all EB snapshots in PC-20",
        "adaptation_note": "Input pcs has 100 columns; this chapter uses the first 20 PCs as PC-20.",
    }
    save_json(OUT_DIR / "eb_data_summary.json", summary)
    return {
        "keys": keys,
        "pcs20_all": pcs20,
        "pcs20_raw_all": pcs20_raw,
        "phate_all": phate,
        "labels": labels,
        "counts": counts,
        "pc_mean": mean,
        "pc_std": std,
        "idx_source": idx_source,
        "idx_target": idx_target,
        "X0_pc": pcs20[idx_source],
        "X1_pc": pcs20[idx_target],
        "X0_phate": phate[idx_source],
        "X1_phate": phate[idx_target],
        "source_time": str(source_time),
        "target_time": str(target_time),
        "summary": summary,
    }

EB = load_eb_data()
print("EB keys:", EB["keys"])
print(EB["counts"])
print("Source/target PC shapes:", EB["X0_pc"].shape, EB["X1_pc"].shape)
print("Summary saved to", OUT_DIR / "eb_data_summary.json")


In [ ]:
# Display-only mapping from PC-20 trajectory points back to PHATE for plotting.
# This is not used for training or endpoint distributional metrics.
pc_to_phate_knn = KNeighborsClassifier(n_neighbors=min(15, len(EB["pcs20_all"])), weights="distance")
pc_to_phate_knn.fit(EB["pcs20_all"], np.arange(len(EB["pcs20_all"])))

def pc_to_phate(points_pc):
    points_pc = np.asarray(points_pc, dtype=np.float32)
    dist, ind = pc_to_phate_knn.kneighbors(points_pc, return_distance=True)
    w = 1.0 / np.clip(dist, 1e-6, None)
    w = w / w.sum(axis=1, keepdims=True)
    return np.einsum("nk,nkd->nd", w, EB["phate_all"][ind])

off_manifold_reference_pc = EB["pcs20_all"]
off_manifold_reference_note = "all available EB snapshots in standardized PC-20"
print(off_manifold_reference_note, off_manifold_reference_pc.shape)


## Exp 1. Independent vs OT Coupling on EB

Train random-product CFM and PC-20 OT-CFM with the same `VelocityMLP(input_dim, hidden=128, layers=4)`, Adam lr `1e-3`, 1500 steps, batch size 256, then evaluate rollouts from source cells against the real target distribution in PC-20.


In [ ]:
X0_eb, X1_eb = EB["X0_pc"], EB["X1_pc"]
X0p_eb, X1p_eb = EB["X0_phate"], EB["X1_phate"]
C_eb_norm, C_eb_scale = compute_cost_matrix(X0_eb, X1_eb, normalize=True)
pi_eb_ot, info_eb_ot = sinkhorn_plan(C_eb_norm, epsilon=SINKHORN_EPSILON, return_info=True)
pi_eb_random = independent_coupling(len(X0_eb), len(X1_eb))
save_npz(CACHE_DIR / "exp1_eb_couplings.npz", C_eb_norm=C_eb_norm, pi_ot=pi_eb_ot, pi_random=pi_eb_random, cost_scale=C_eb_scale)
save_json(CACHE_DIR / "exp1_sinkhorn_info.json", info_eb_ot)
print("OT Sinkhorn info:", info_eb_ot)


In [ ]:
def train_or_load_model(name: str, X0, X1, pi, steps: int = TRAINING_STEPS, seed: int = DEFAULT_SEED):
    cache_tag = f"d{X0.shape[1]}_steps{int(steps)}_batch{BATCH_SIZE}_seed{int(seed)}"
    ckpt = CACHE_DIR / f"{name}_{cache_tag}_model.pt"
    hist_path = CACHE_DIR / f"{name}_{cache_tag}_history.csv"
    model = VelocityMLP(input_dim=X0.shape[1], hidden=128, layers=4).to(DEVICE)
    if ckpt.exists():
        payload = load_pt(ckpt, map_location=DEVICE)
        if int(payload.get("input_dim", -1)) != int(X0.shape[1]) or int(payload.get("steps", -1)) != int(steps):
            raise ValueError(f"Checkpoint metadata mismatch for {ckpt}")
        model.load_state_dict(payload["state_dict"])
        history = pd.read_csv(hist_path) if hist_path.exists() else pd.DataFrame()
        print(f"Loaded {name} from {ckpt}")
        return model, history
    sampler = PlanPairSampler(X0, X1, pi=pi, seed=seed)
    history = train_cfm(model, sampler, steps=steps, batch_size=BATCH_SIZE, lr=1e-3, device=DEVICE, seed=seed)
    save_pt(ckpt, {"state_dict": model.state_dict(), "input_dim": int(X0.shape[1]), "seed": int(seed), "steps": int(steps), "batch_size": int(BATCH_SIZE)})
    save_csv(hist_path, history)
    print(f"Trained {name}; parameters={count_parameters(model)}; final loss={history.loss.iloc[-1]:.4f}")
    return model, history

random_model, random_hist = train_or_load_model("exp1_random_cfm", X0_eb, X1_eb, pi_eb_random, seed=42)
ot_model, ot_hist = train_or_load_model("exp1_ot_cfm", X0_eb, X1_eb, pi_eb_ot, seed=42)


In [ ]:
def midpoint_direction_dispersion(X0, X1, pi, n_pairs: int = 4096, k: int = 25, seed: int = DEFAULT_SEED):
    """Neighborhood dispersion of sampled chord directions around pair midpoints."""
    idx0, idx1 = sample_from_plan(pi, n_pairs, seed=seed)
    x0, x1 = X0[idx0], X1[idx1]
    chords = x1 - x0
    chord_norm = np.linalg.norm(chords, axis=1)
    directions = chords / np.clip(chord_norm[:, None], 1e-12, None)
    midpoints = 0.5 * (x0 + x1)
    n_neighbors = max(2, min(int(k), len(midpoints)))
    nn = NearestNeighbors(n_neighbors=n_neighbors).fit(midpoints)
    neigh = nn.kneighbors(midpoints, return_distance=False)
    angular_std = []
    for ids in neigh:
        local_dirs = directions[ids]
        mean_dir = local_dirs.mean(axis=0)
        mean_norm = np.linalg.norm(mean_dir)
        if mean_norm < 1e-12:
            angles = np.full(len(ids), 90.0)
        else:
            mean_dir = mean_dir / mean_norm
            cosang = np.clip(local_dirs @ mean_dir, -1.0, 1.0)
            angles = np.degrees(np.arccos(cosang))
        angular_std.append(float(np.std(angles)))
    stats = pd.DataFrame({
        "idx0": idx0,
        "idx1": idx1,
        "pc20_chord_length": chord_norm,
        "midpoint_direction_angular_std_deg": angular_std,
    })
    return stats

exp1_rows = []
exp1_paths = {}
pair_stats_exp1 = {}
for name, model, pi in [("random_cfm", random_model, pi_eb_random), ("ot_cfm", ot_model, pi_eb_ot)]:
    z, traj, times = trajectory_rollout(model, X0_eb, nfe=DEFAULT_NFE, return_path=True)
    exp1_paths[name] = traj
    metrics = evaluate_endpoint(z, X1_eb)
    pair_stats = midpoint_direction_dispersion(X0_eb, X1_eb, pi, n_pairs=4096, k=25, seed=44)
    pair_stats["method"] = name
    pair_stats_exp1[name] = pair_stats
    chord = pair_stats["pc20_chord_length"].to_numpy()
    disp = pair_stats["midpoint_direction_angular_std_deg"].to_numpy()
    row = {
        "method": name,
        **metrics,
        "mean_pc20_chord_length": float(chord.mean()),
        "median_pc20_chord_length": float(np.median(chord)),
        "std_pc20_chord_length": float(chord.std()),
        "midpoint_direction_angular_std_mean_deg": float(disp.mean()),
        "midpoint_direction_angular_std_median_deg": float(np.median(disp)),
        "midpoint_direction_angular_std_iqr_deg": float(np.quantile(disp, 0.75) - np.quantile(disp, 0.25)),
        "training_space": "standardized PC-20",
        "cost_space": "standardized PC-20" if name == "ot_cfm" else "independent product coupling",
        "display_space": "PHATE 2D only for figures",
        "endpoint_metric_space": "standardized PC-20",
    }
    exp1_rows.append(row)
    save_npz(CACHE_DIR / f"exp1_{name}_rollout.npz", endpoint=z, trajectory=traj, times=times)

exp1_pair_stats = pd.concat(pair_stats_exp1.values(), ignore_index=True)
save_csv(CACHE_DIR / "exp1_pair_level_statistics.csv", exp1_pair_stats)
exp1_metrics = pd.DataFrame(exp1_rows)
save_csv(OUT_DIR / "exp1_metrics.csv", exp1_metrics)
exp1_metrics


In [ ]:
# Figures requested for Exp 1.
idx_r0, idx_r1 = sample_from_plan(pi_eb_random, 900, seed=50)
idx_o0, idx_o1 = sample_from_plan(pi_eb_ot, 900, seed=51)
r_chord = np.linalg.norm(X1_eb[idx_r1] - X0_eb[idx_r0], axis=1)
o_chord = np.linalg.norm(X1_eb[idx_o1] - X0_eb[idx_o0], axis=1)
plot_pair_panels(
    X0p_eb,
    X1p_eb,
    [
        {"title": "Random product pairs", "idx0": idx_r0, "idx1": idx_r1, "values": r_chord, "seed": 50, "max_lines": 140},
        {"title": "PC-20 OT pairs", "idx0": idx_o0, "idx1": idx_o1, "values": o_chord, "seed": 51, "max_lines": 140},
    ],
    "fig4_2_random_vs_ot_pairs.png",
    "Endpoint pair choices displayed in PHATE; line color is PC-20 chord length",
    value_label="standardized PC-20 chord length",
)

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.0))
for method, color in [("random_cfm", PALETTE["random"]), ("ot_cfm", PALETTE["ot"] )]:
    stats = pair_stats_exp1[method]
    axes[0].hist(stats["pc20_chord_length"], bins=45, alpha=0.55, color=color, label=method.replace("_", " "))
axes[0].set_title("PC-20 chord length")
axes[0].set_xlabel("standardized PC-20 distance")
axes[0].set_ylabel("sampled pairs")
axes[0].legend(frameon=False)

box_data = [
    pair_stats_exp1["random_cfm"]["midpoint_direction_angular_std_deg"].to_numpy(),
    pair_stats_exp1["ot_cfm"]["midpoint_direction_angular_std_deg"].to_numpy(),
]
axes[1].boxplot(box_data, labels=["random", "OT"], showfliers=False)
axes[1].set_title("Midpoint-neighborhood direction dispersion")
axes[1].set_ylabel("angular std around local mean direction (deg)")
save_figure(fig, "fig4_1_supp_pair_statistics.png")

plot_projected_trajectories(
    {"random": exp1_paths["random_cfm"], "ot": exp1_paths["ot_cfm"]},
    X0p_eb,
    X1p_eb,
    pc_to_phate,
    "fig4_5_random_vs_ot_projected_trajectories.png",
    "Euler rollouts projected to PHATE with local neighborhood arrows",
)
plot_projected_trajectories(
    {"random": exp1_paths["random_cfm"], "ot": exp1_paths["ot_cfm"]},
    X0p_eb,
    X1p_eb,
    pc_to_phate,
    "fig4_1_independent_coupling_paths.png",
    "Independent versus OT-CFM paths (PHATE display only)",
)


## Exp 2. Sinkhorn Epsilon + Minibatch Ablation

Part A varies Sinkhorn epsilon on the median-normalized PC-20 cost.  Part B estimates minibatch-coupling variability by repeatedly recomputing Sinkhorn on random source/target subsets.  PHATE pair distances are display diagnostics only.


In [ ]:
eps_rows = []
pi_by_eps = {}
for eps in EPSILON_GRID:
    pi, info = sinkhorn_plan(C_eb_norm, epsilon=eps, return_info=True)
    pi_by_eps[float(eps)] = pi
    idx0, idx1 = sample_from_plan(pi, 4096, seed=100 + int(eps * 1000))
    diag = coupling_diagnostics(pi, C_eb_norm)
    eps_rows.append({
        "epsilon": float(eps),
        "expected_cost_normalized": float(diag["expected_cost"]),
        "cost_scale": float(C_eb_scale),
        "entropy": float(diag["entropy"]),
        "effective_support": float(diag["effective_support"]),
        "sinkhorn_converged": bool(info["sinkhorn_converged"]),
        "sinkhorn_n_iter": int(info["n_iter"]),
        "row_l1_error": float(info["row_l1_error"]),
        "col_l1_error": float(info["col_l1_error"]),
        "warning_message": info.get("warning_message", ""),
        "mean_pair_distance_pc20": float(np.linalg.norm(X1_eb[idx1] - X0_eb[idx0], axis=1).mean()),
        "mean_pair_distance_phate_display_only": float(np.linalg.norm(X1p_eb[idx1] - X0p_eb[idx0], axis=1).mean()),
        "cost_space": "standardized PC-20 median-normalized",
        "phate_distance_role": "display diagnostic only",
    })
eps_table = pd.DataFrame(eps_rows)
save_csv(OUT_DIR / "table4_A_sinkhorn_epsilon_ablation.csv", eps_table)
save_npz(CACHE_DIR / "exp2_epsilon_plans.npz", **{f"pi_eps_{str(k).replace('.', '_')}": v for k, v in pi_by_eps.items()})
eps_table


In [ ]:
# Epsilon pair visualization.
panels = []
for eps in [0.01, 0.05, 0.1, 0.5]:
    pi = pi_by_eps[float(eps)]
    idx0, idx1 = sample_from_plan(pi, 900, seed=120 + int(eps * 1000))
    panels.append({"title": f"epsilon={eps:g}", "idx0": idx0, "idx1": idx1, "color": PALETTE["ot"], "seed": 2})
plot_pair_panels(
    X0p_eb, X1p_eb, panels,
    "fig4_2b_epsilon_ablation_pairs.png",
    "Sinkhorn epsilon changes PC-20 coupling; displayed in PHATE only",
)


In [ ]:
minibatch_rows = []
mb_sizes = [64, 128, 256, 512, "full"]
rng = np.random.default_rng(202)
for mb in mb_sizes:
    repeats = 1 if mb == "full" else 10
    for rep in range(repeats):
        if mb == "full":
            x0_sub, x1_sub = X0_eb, X1_eb
        else:
            i0 = np.sort(rng.choice(len(X0_eb), size=min(int(mb), len(X0_eb)), replace=False))
            i1 = np.sort(rng.choice(len(X1_eb), size=min(int(mb), len(X1_eb)), replace=False))
            x0_sub, x1_sub = X0_eb[i0], X1_eb[i1]
        C_sub, scale_sub = compute_cost_matrix(x0_sub, x1_sub, normalize=True)
        pi_sub, info_sub = sinkhorn_plan(C_sub, epsilon=SINKHORN_EPSILON, return_info=True)
        diag = coupling_diagnostics(pi_sub, C_sub)
        minibatch_rows.append({
            "minibatch_size": str(mb),
            "repeat": int(rep),
            "n_source": int(len(x0_sub)),
            "n_target": int(len(x1_sub)),
            "expected_cost_normalized": float(diag["expected_cost"]),
            "effective_support": float(diag["effective_support"]),
            "entropy": float(diag["entropy"]),
            "sinkhorn_converged": bool(info_sub["sinkhorn_converged"]),
            "cost_scale": float(scale_sub),
        })
minibatch_raw = pd.DataFrame(minibatch_rows)
minibatch_summary = minibatch_raw.groupby("minibatch_size", sort=False).agg(
    repeats=("repeat", "count"),
    expected_cost_mean=("expected_cost_normalized", "mean"),
    expected_cost_std=("expected_cost_normalized", "std"),
    effective_support_mean=("effective_support", "mean"),
    effective_support_std=("effective_support", "std"),
).reset_index()
save_csv(OUT_DIR / "tableA_4_1_minibatch_ablation.csv", minibatch_summary)
save_csv(CACHE_DIR / "exp2_minibatch_ablation_raw.csv", minibatch_raw)
minibatch_summary


## Exp 3. Rectified Flow

Use the Exp 1 OT-CFM as the base model.  Reflow is trained against induced endpoints from model rollouts, but endpoint metrics are always computed against the real EB target distribution in standardized PC-20.


In [ ]:
def train_reflow_round(name: str, base_model, X0, steps: int = TRAINING_STEPS, seed: int = DEFAULT_SEED):
    endpoint_path = CACHE_DIR / f"{name}_induced_endpoint.npz"
    if endpoint_path.exists():
        Z = load_npz(endpoint_path)["endpoint"]
    else:
        Z = rollout_euler(base_model, X0, nfe=DEFAULT_NFE)
        save_npz(endpoint_path, endpoint=Z)
    pi_diag = np.eye(len(X0), len(Z), dtype=float)
    pi_diag /= pi_diag.sum()
    model, hist = train_or_load_model(name, X0, Z, pi_diag, steps=steps, seed=seed)
    return model, Z, hist

reflow1_model, Z1_1, reflow1_hist = train_reflow_round("exp3_reflow1", ot_model, X0_eb, seed=43)
reflow2_model, Z1_2, reflow2_hist = train_reflow_round("exp3_reflow2", reflow1_model, X0_eb, seed=44)


In [ ]:
models_exp3 = {
    "random_cfm": random_model,
    "ot_cfm": ot_model,
    "reflow_1": reflow1_model,
    "reflow_2": reflow2_model,
}
traj_exp3 = {}
rows = []
for name, model in models_exp3.items():
    z, traj, times = trajectory_rollout(model, X0_eb, nfe=DEFAULT_NFE, return_path=True)
    traj_exp3[name] = traj
    rows.append({
        "method": name,
        **evaluate_endpoint(z, X1_eb),
        "straightness_action_S": straightness_action_S(traj, times),
        "tortuosity_straightness": tortuosity_straightness(traj),
        "path_length": path_length(traj),
        "path_energy": path_energy(traj, times),
        "coarse_step_error_nfe4_vs_nfe64": coarse_step_error(model, X0_eb, nfe_coarse=4, nfe_fine=64),
        "endpoint_reference": "real EB target distribution in standardized PC-20",
    })
reflow_table = pd.DataFrame(rows)
save_csv(OUT_DIR / "table4_1_reflow_ablation.csv", reflow_table)

nfe_rows = []
for name, model in models_exp3.items():
    for nfe in NFE_GRID:
        z = rollout_euler(model, X0_eb, nfe=nfe)
        nfe_rows.append({"method": name, "nfe": int(nfe), **evaluate_endpoint(z, X1_eb)})
nfe_table = pd.DataFrame(nfe_rows)
save_csv(CACHE_DIR / "exp3_nfe_sensitivity.csv", nfe_table)
reflow_table


In [ ]:
plot_projected_trajectories(
    {"ot": traj_exp3["ot_cfm"], "reflow1": traj_exp3["reflow_1"], "reflow2": traj_exp3["reflow_2"]},
    X0p_eb, X1p_eb, pc_to_phate,
    "fig4_4_reflow_trajectories.png",
    "Reflow paths projected to PHATE; straightness is not biological validation",
)
plot_metric_lines(
    nfe_table, x="nfe", y="sliced_w2", hue="method",
    filename="fig4_4b_nfe_endpoint_error.png",
    title="NFE sensitivity against real EB target in PC-20",
)


## Exp 4. Coupling Diagnostic Table

For random CFM, OT-CFM, reflow-1, and reflow-2, compute endpoint, path geometry, off-manifold, and coarse-step diagnostics.  The off-manifold reference is all observed EB snapshots in standardized PC-20 when available.


In [ ]:
import gc

rows = []
exp4_progress_path = CACHE_DIR / "exp4_path_diagnostics_progress.json"
for name, model in models_exp3.items():
    save_json(exp4_progress_path, {"stage": "starting", "method": name, "completed_methods": [r["method"] for r in rows]})
    z, traj, times = trajectory_rollout(model, X0_eb, nfe=DEFAULT_NFE, return_path=True)
    row = {
        "method": name,
        **evaluate_endpoint(z, X1_eb),
        "path_length": path_length(traj),
        "path_energy_action": path_energy(traj, times),
        "straightness_action_S": straightness_action_S(traj, times),
        "tortuosity_straightness": tortuosity_straightness(traj),
        "off_manifold_knn_k15": off_manifold_knn(traj, off_manifold_reference_pc, k=15),
        "coarse_step_error_nfe4_vs_nfe64": coarse_step_error(model, X0_eb, nfe_coarse=4, nfe_fine=64),
        "off_manifold_reference": off_manifold_reference_note,
        "training_space": "standardized PC-20",
        "endpoint_metric_space": "standardized PC-20 real target",
    }
    rows.append(row)
    save_csv(OUT_DIR / "table4_1_path_geometry_diagnostics.csv", pd.DataFrame(rows))
    save_json(exp4_progress_path, {"stage": "completed", "method": name, "completed_methods": [r["method"] for r in rows]})
    del z, traj
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
path_diag_table = pd.DataFrame(rows)
save_csv(OUT_DIR / "table4_1_path_geometry_diagnostics.csv", path_diag_table)
plot_projected_trajectories(
    {"random": traj_exp3["random_cfm"], "ot": traj_exp3["ot_cfm"]},
    X0p_eb, X1p_eb, pc_to_phate,
    "fig4_5_random_vs_ot_projected_trajectories.png",
    "Random and OT trajectories projected to PHATE with local arrows",
)
path_diag_table


## Artifact Manifest

This final section verifies the figures, tables, JSON files, and split-specific manifest generated by this notebook.


In [ ]:
expected_figures = ['fig4_1_independent_coupling_paths.png', 'fig4_2_random_vs_ot_pairs.png', 'fig4_1_supp_pair_statistics.png', 'fig4_2b_epsilon_ablation_pairs.png', 'fig4_4_reflow_trajectories.png', 'fig4_5_random_vs_ot_projected_trajectories.png', 'fig4_4b_nfe_endpoint_error.png']
expected_tables = ['eb_data_summary.json', 'exp1_metrics.csv', 'table4_A_sinkhorn_epsilon_ablation.csv', 'tableA_4_1_minibatch_ablation.csv', 'table4_1_reflow_ablation.csv', 'table4_1_path_geometry_diagnostics.csv']

run_config = {
    "SMOKE_MODE": bool(SMOKE_MODE),
    "TRAINING_STEPS": int(TRAINING_STEPS),
    "BATCH_SIZE": int(BATCH_SIZE),
    "DEFAULT_NFE": int(DEFAULT_NFE),
    "DEVICE": str(DEVICE),
}
save_json(OUT_DIR / "run_config_04_1_coupling_geometry.json", run_config)

manifest_rows = []
for key, value in run_config.items():
    manifest_rows.append({"artifact": f"RUN_CONFIG:{key}={value}", "kind": "run_config", "exists": True, "bytes": 0})
for name in expected_figures:
    p = FIG_DIR / name
    manifest_rows.append({"artifact": str(p.relative_to(PROJECT_ROOT)), "kind": "figure", "exists": p.exists(), "bytes": p.stat().st_size if p.exists() else 0})
for name in expected_tables:
    p = OUT_DIR / name
    manifest_rows.append({"artifact": str(p.relative_to(PROJECT_ROOT)), "kind": "table_or_json", "exists": p.exists(), "bytes": p.stat().st_size if p.exists() else 0})
artifact_manifest = pd.DataFrame(manifest_rows)
save_csv(OUT_DIR / "artifact_manifest_04_1_coupling_geometry.csv", artifact_manifest)

missing = artifact_manifest.loc[(artifact_manifest["kind"] != "run_config") & (~artifact_manifest["exists"])]
if not missing.empty:
    raise FileNotFoundError("Missing expected Chapter 4 split artifacts: " + ", ".join(missing["artifact"].tolist()))

print("Artifact manifest")
display(artifact_manifest)
